In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
 

In [31]:
players = pd.read_csv("C:/Users/laila/Documents/Football Project/archive (4)/male_players.csv")

In [32]:
players['Nation'] = players['Nation'].replace({'Korea Republic': 'South Korea'})

In [33]:
top20 = pd.read_csv("C:/Users/laila/Documents/Football Project/top20_contenders.csv")

In [34]:
top20_nations = top20['TEAM'].tolist()

In [35]:
print("Top 20 nations:", top20_nations)
print("Total players in dataset:", len(players))

Top 20 nations: ['Spain', 'France', 'England', 'Argentina', 'Netherlands', 'Australia', 'Brazil', 'Portugal', 'South Korea', 'Iran', 'Germany', 'Japan', 'Morocco', 'Croatia', 'Mexico', 'United States', 'Italy', 'New Zealand', 'Senegal', 'Nigeria']
Total players in dataset: 16161


In [36]:
players_top20 = players[players['Nation'].isin(top20_nations)].copy()

In [37]:
print(f"Players from top 20 nations: {len(players_top20)}")

Players from top 20 nations: 8024


In [38]:
matched = players_top20['Nation'].unique().tolist()

In [39]:
unmatched = [n for n in top20_nations if n not in matched]
if unmatched:
    print(f"\n⚠️  These nations had NO players matched: {unmatched}")
    print("Check spelling vs dataset:", sorted(players['Nation'].unique()))
else:
    print("\n✓ All top 20 nations matched successfully")


✓ All top 20 nations matched successfully


In [40]:
def get_top23(group):
    return group.nlargest(23, 'OVR')

In [45]:
top23_list = []
for nation in top20_nations:
    nation_players = players_top20[players_top20['Nation'] == nation]
    top23 = nation_players.nlargest(23, 'OVR')
    top23_list.append(top23)

squads = pd.concat(top23_list, ignore_index=True)

In [46]:
print(f"\nTotal squad players (top 23 per nation): {len(squads)}")
print(squads.groupby('Nation').size())


Total squad players (top 23 per nation): 435
Nation
Argentina        23
Australia        23
Brazil           23
Croatia          23
England          23
France           23
Germany          23
Iran              9
Italy            23
Japan            23
Mexico           23
Morocco          23
Netherlands      12
New Zealand      23
Nigeria          23
Portugal         23
Senegal          23
South Korea      23
Spain            23
United States    23
dtype: int64


In [48]:
def squad_stability(group):
    ages = group['Age']
    return pd.Series({
        'avg_age'     : round(ages.mean(), 1),
        'age_std'     : round(ages.std(), 1), 
        'peak_count'  : ((ages >= 24) & (ages <= 29)).sum(), 
        'aging_count' : (ages >= 30).sum(),
        'young_count' : (ages < 24).sum(), 
        'youngest'    : int(ages.min()),
        'oldest'      : int(ages.max()),
    })
stability = squads.groupby('Nation').apply(squad_stability).reset_index()
print("\n=== SQUAD STABILITY (top 23 by OVR) ===")
print(stability.sort_values('avg_age').to_string(index=False))


=== SQUAD STABILITY (top 23 by OVR) ===
       Nation  avg_age  age_std  peak_count  aging_count  young_count  youngest  oldest
United States     24.7      3.3        10.0          3.0         10.0      21.0    34.0
       Mexico     26.1      4.2        11.0          4.0          8.0      19.0    34.0
  New Zealand     26.9      4.6        10.0          6.0          7.0      19.0    36.0
      Nigeria     27.0      3.1        18.0          3.0          2.0      23.0    35.0
        Japan     27.2      2.6        19.0          3.0          1.0      23.0    34.0
      England     27.3      3.9        12.0          7.0          4.0      21.0    34.0
      Senegal     27.3      4.2        10.0          8.0          5.0      20.0    34.0
       Brazil     27.5      3.3        12.0          8.0          3.0      20.0    33.0
       France     27.7      4.2        16.0          5.0          2.0      21.0    37.0
  Netherlands     27.7      5.1         6.0          4.0          2.0      21.0

In [51]:
def ovr_analysis(group):
    ovr = group['OVR']
    return pd.Series({
        'avg_ovr'          : round(ovr.mean(), 1),
        'best_player_ovr'  : int(ovr.max()),
        'best_player'      : group.loc[ovr.idxmax(), 'Name'],
        'elite_count'      : (ovr >= 80).sum(),    # OVR 80+ = elite
        'worldclass_count' : (ovr >= 85).sum(),    # OVR 85+ = world class
    })

In [52]:
ovr_summary = squads.groupby('Nation').apply(ovr_analysis).reset_index()
ovr_summary = ovr_summary.sort_values('avg_ovr', ascending=False).reset_index(drop=True)
ovr_summary['rank'] = ovr_summary.index + 1
 
print("\n=== OVERALL RATINGS BY NATION ===")
print(ovr_summary.to_string(index=False))


=== OVERALL RATINGS BY NATION ===
       Nation  avg_ovr  best_player_ovr           best_player  elite_count  worldclass_count  rank
       France     84.7               91         Kylian Mbappé           23                10     1
      England     84.5               90       Jude Bellingham           23                10     2
       Brazil     84.4               90              Vini Jr.           23                10     3
      Germany     84.3               89 Marc-André ter Stegen           23                11     4
        Spain     84.1               91                 Rodri           23                 7     5
     Portugal     83.5               88            Rúben Dias           23                 9     6
    Argentina     83.5               89      Lautaro Martínez           23                 5     7
        Italy     83.3               89  Gianluigi Donnarumma           23                 4     8
      Croatia     78.3               86           Luka Modrić            8

In [54]:
skill_cols = ['PAC', 'SHO', 'PAS', 'DRI', 'DEF', 'PHY']
skills = squads.groupby('Nation')[skill_cols].mean().round(1).reset_index()
skills = skills.sort_values('PAC', ascending=False)

In [55]:
print("\n=== CORE SKILL RATINGS BY NATION (avg of top 23) ===")
print(skills.to_string(index=False))
 
print("\n--- Best nation per skill ---")
for skill in skill_cols:
    best_nation = skills.loc[skills[skill].idxmax(), 'Nation']
    best_val    = skills[skill].max()
    print(f"  {skill}: {best_nation} ({best_val})")


=== CORE SKILL RATINGS BY NATION (avg of top 23) ===
       Nation  PAC  SHO  PAS  DRI  DEF  PHY
      Nigeria 80.7 68.5 64.9 74.8 47.6 74.3
       France 80.7 70.2 76.3 80.9 63.5 76.5
       Brazil 78.8 72.4 77.1 81.1 62.6 76.1
United States 78.0 61.4 67.7 74.3 61.4 72.3
        Japan 77.9 63.1 68.7 74.3 58.9 69.0
      England 77.0 71.3 79.1 81.4 65.4 75.0
        Italy 76.7 70.0 74.9 79.9 67.5 78.8
     Portugal 76.1 72.4 78.7 81.7 63.1 74.7
      Senegal 75.7 64.5 66.4 73.8 56.0 72.2
  South Korea 75.5 68.1 67.4 74.0 50.3 71.8
    Argentina 75.0 73.7 77.3 80.6 63.0 76.4
        Spain 74.8 72.2 79.7 81.5 65.3 74.5
      Morocco 74.3 68.4 71.1 76.7 53.4 72.0
      Germany 73.7 74.3 78.3 79.5 63.8 77.2
  Netherlands 72.3 55.0 59.8 67.8 48.8 60.9
         Iran 71.6 58.9 63.0 68.0 54.0 70.4
       Mexico 71.5 58.2 61.9 67.9 54.2 68.9
      Croatia 71.0 68.8 72.8 77.7 59.8 73.7
    Australia 70.5 60.0 63.7 67.6 58.0 73.0
  New Zealand 68.7 58.1 61.3 65.5 54.5 70.2

--- Best nation per s

In [56]:
position_groups = {
    'Goalkeeper' : ['GK'],
    'Defender'   : ['CB', 'LB', 'RB'],
    'Midfielder' : ['CM', 'CAM', 'CDM', 'LM', 'RM'],
    'Winger'     : ['LW', 'RW'],
    'Forward'    : ['ST'],
}
 

In [57]:
squads['role'] = squads['Position'].map(
    {pos: role for role, positions in position_groups.items() for pos in positions}
)

In [58]:
def top5_avg_ovr(group):
    return round(group.nlargest(5, 'OVR')['OVR'].mean(), 1)
 
role_strength = (
    squads.groupby(['Nation', 'role'])
          .apply(top5_avg_ovr)
          .reset_index()
          .rename(columns={0: 'avg_ovr_top5'})
)
 

In [61]:
role_pivot = role_strength.pivot(
    index='Nation', columns='role', values='avg_ovr_top5').reset_index()
role_pivot.columns.name = None

In [62]:
print("\n=== POSITION STRENGTH (avg OVR of top 5 per role) ===")
print(role_pivot.sort_values('Forward', ascending=False).to_string(index=False))
 
print("\n--- Best nation per position ---")
for col in ['Goalkeeper', 'Defender', 'Midfielder', 'Winger', 'Forward']:
    if col in role_pivot.columns:
        best = role_pivot.loc[role_pivot[col].idxmax(), 'Nation']
        val  = role_pivot[col].max()
        print(f"  {col}: {best} ({val})")
 


=== POSITION STRENGTH (avg OVR of top 5 per role) ===
       Nation  Defender  Forward  Goalkeeper  Midfielder  Winger
      England      84.4     87.5        83.0        85.8    85.2
       France      85.4     86.2        87.0        84.2    86.0
     Portugal      84.2     85.5        82.5        85.8    83.0
    Argentina      82.8     84.5        82.8        84.8    88.0
        Spain      83.4     83.3        84.3        86.6     NaN
      Germany      85.4     82.5        84.8        86.4     NaN
       Brazil      85.2     82.0        88.5        83.2    86.0
        Italy      84.0     82.0        84.5        84.0    83.3
      Nigeria      75.2     81.0         NaN        76.6    80.0
         Iran      70.0     80.0        71.0        67.2     NaN
      Morocco      78.8     79.2        84.0        79.0    76.0
      Croatia      77.6     77.4        78.0        82.2    76.0
      Senegal      77.0     76.8        77.5        77.8    76.0
  South Korea      77.5     76.6   

In [63]:
player_summary = ovr_summary.merge(stability,  on='Nation', how='left')
player_summary = player_summary.merge(skills,   on='Nation', how='left')
player_summary = player_summary.merge(role_pivot, on='Nation', how='left')

In [64]:
print("\n=== MASTER PLAYER SUMMARY ===")
print(player_summary[[
    'rank', 'Nation', 'avg_ovr', 'elite_count', 'worldclass_count',
    'avg_age', 'age_std', 'peak_count', 'aging_count', 'young_count',
    'PAC', 'SHO', 'PAS', 'DRI', 'DEF', 'PHY'
]].to_string(index=False))


=== MASTER PLAYER SUMMARY ===
 rank        Nation  avg_ovr  elite_count  worldclass_count  avg_age  age_std  peak_count  aging_count  young_count  PAC  SHO  PAS  DRI  DEF  PHY
    1        France     84.7           23                10     27.7      4.2        16.0          5.0          2.0 80.7 70.2 76.3 80.9 63.5 76.5
    2       England     84.5           23                10     27.3      3.9        12.0          7.0          4.0 77.0 71.3 79.1 81.4 65.4 75.0
    3        Brazil     84.4           23                10     27.5      3.3        12.0          8.0          3.0 78.8 72.4 77.1 81.1 62.6 76.1
    4       Germany     84.3           23                11     29.5      4.1        10.0         11.0          2.0 73.7 74.3 78.3 79.5 63.8 77.2
    5         Spain     84.1           23                 7     28.3      4.1        12.0          8.0          3.0 74.8 72.2 79.7 81.5 65.3 74.5
    6      Portugal     83.5           23                 9     28.0      3.8        13.0    

In [65]:
player_summary.to_csv(
    "C:/Users/laila/Documents/Football Project/player_summary_top20.csv",
    index=False
)
print("\nSaved: player_summary_top20.csv")


Saved: player_summary_top20.csv


In [68]:
print(top20.columns.tolist())
print(top20[['TEAM', 'confederation']].to_string())

['TEAM', 'matches_played', 'goals_scored', 'goals_conceded', 'clean_sheets', 'wins', 'draws', 'losses', 'goals_per_game', 'goals_conceded_per_game', 'goal_diff_per_game', 'cleansheet_pct', 'win_rate', 'wc_appearances', 'recent_win_rate', 'wc2022_stage', 'composite_score', 'rank', 'confederation']
             TEAM confederation
0           Spain          UEFA
1          France          UEFA
2         England          UEFA
3       Argentina      CONMEBOL
4     Netherlands          UEFA
5       Australia           AFC
6          Brazil      CONMEBOL
7        Portugal          UEFA
8     South Korea           AFC
9            Iran           AFC
10        Germany          UEFA
11          Japan           AFC
12        Morocco           CAF
13        Croatia          UEFA
14         Mexico      CONCACAF
15  United States      CONCACAF
16          Italy          UEFA
17    New Zealand           OFC
18        Senegal           CAF
19        Nigeria           CAF


In [69]:
confederation_map = {
    'Spain': 'UEFA', 'Germany': 'UEFA', 'England': 'UEFA', 'France': 'UEFA',
    'Netherlands': 'UEFA', 'Portugal': 'UEFA', 'Italy': 'UEFA', 'Belgium': 'UEFA',
    'Croatia': 'UEFA', 'Switzerland': 'UEFA', 'Poland': 'UEFA',
    'Brazil': 'CONMEBOL', 'Argentina': 'CONMEBOL', 'Uruguay': 'CONMEBOL',
    'Colombia': 'CONMEBOL', 'Chile': 'CONMEBOL',
    'Mexico': 'CONCACAF', 'United States': 'CONCACAF',
    'Japan': 'AFC', 'South Korea': 'AFC', 'Australia': 'AFC',
    'Iran': 'AFC', 'Saudi Arabia': 'AFC',
    'Morocco': 'CAF', 'Senegal': 'CAF', 'Nigeria': 'CAF',
    'Ghana': 'CAF', 'Cameroon': 'CAF',
    'New Zealand': 'OFC',
}

top20['confederation'] = top20['TEAM'].map(confederation_map).fillna('Other')

# Re-export with confederation included
top20.to_csv("C:/Users/laila/Documents/Football Project/top20_contenders.csv", index=False)

print(top20[['TEAM', 'confederation']].to_string())

             TEAM confederation
0           Spain          UEFA
1          France          UEFA
2         England          UEFA
3       Argentina      CONMEBOL
4     Netherlands          UEFA
5       Australia           AFC
6          Brazil      CONMEBOL
7        Portugal          UEFA
8     South Korea           AFC
9            Iran           AFC
10        Germany          UEFA
11          Japan           AFC
12        Morocco           CAF
13        Croatia          UEFA
14         Mexico      CONCACAF
15  United States      CONCACAF
16          Italy          UEFA
17    New Zealand           OFC
18        Senegal           CAF
19        Nigeria           CAF


In [1]:
import pandas as pd

flags = {
    'Spain': '🇪🇸', 'England': '🏴󠁧󠁢󠁥󠁮󠁧󠁿', 'France': '🇫🇷',
    'Argentina': '🇦🇷', 'Brazil': '🇧🇷', 'Portugal': '🇵🇹',
    'Netherlands': '🇳🇱', 'Germany': '🇩🇪', 'Morocco': '🇲🇦',
    'Croatia': '🇭🇷', 'Japan': '🇯🇵', 'Belgium': '🇧🇪',
    'Iran': '🇮🇷', 'Australia': '🇦🇺', 'Senegal': '🇸🇳',
    'South Korea': '🇰🇷', 'Mexico': '🇲🇽', 'Ivory Coast': '🇨🇮',
    'United States': '🇺🇸', 'Switzerland': '🇨🇭', 'Algeria': '🇩🇿',
    'Egypt': '🇪🇬', 'Tunisia': '🇹🇳', 'Norway': '🇳🇴',
    'Uruguay': '🇺🇾', 'New Zealand': '🇳🇿', 'Uzbekistan': '🇺🇿',
    'Turkey': '🇹🇷', 'Ghana': '🇬🇭', 'Canada': '🇨🇦',
    'Sweden': '🇸🇪', 'Colombia': '🇨🇴', 'Austria': '🇦🇹',
    'Czech Republic': '🇨🇿', 'Jordan': '🇯🇴', 'Saudi Arabia': '🇸🇦',
    'Scotland': '🏴󠁧󠁢󠁳󠁣󠁴󠁿', 'Haiti': '🇭🇹', 'DR Congo': '🇨🇩',
    'South Africa': '🇿🇦', 'Qatar': '🇶🇦', 'Panama': '🇵🇦',
    'Cape Verde': '🇨🇻', 'Ecuador': '🇪🇨',
    'Bosnia and Herzegovina': '🇧🇦', 'Iraq': '🇮🇶',
    'Curacao': '🇨🇼', 'Paraguay': '🇵🇾',
}

# Check actual column names in each file first
files = [
    'wc2026_all48_summary',
    'monte_carlo_results_2026',
    'top20_contenders',
    'group_predictions',
    'stage_progression',
]

for filename in files:
    path = f"C:/Users/laila/Documents/Football Project/{filename}.csv"
    df = pd.read_csv(path)
    
    # Auto detect team column
    team_col = None
    for col in df.columns:
        if col.upper() == 'TEAM':
            team_col = col
            break
    
    if team_col is None:
        print(f"✗ {filename} — no TEAM column found. Columns: {df.columns.tolist()}")
        continue
    
    df['Flag'] = df[team_col].map(flags).fillna('')
    df['Team with Flag'] = df['Flag'] + ' ' + df[team_col]
    df.to_csv(path, index=False)
    print(f"✓ {filename} — used column '{team_col}'")

print("\nAll done!")

✓ wc2026_all48_summary — used column 'TEAM'
✓ monte_carlo_results_2026 — used column 'TEAM'
✓ top20_contenders — used column 'TEAM'
✓ group_predictions — used column 'Team'
✓ stage_progression — used column 'TEAM'

All done!
